In [ ]:
! pip install scikit-allel
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm # For color maps
import sklearn.metrics
import tqdm
import allel

In [ ]:
# Downloading all files
REMOTE_BUCKET = ""
! gcloud storage cp "{REMOTE_BUCKET}/v3/15x/workpackage_1_ultralong_annotated/merged/scored/*_all_score.vcf.gz*" .
! gcloud storage cp "{REMOTE_BUCKET}/v3/ultralong_inputs/GRCh38_HG2-T2TQ100-V1.1_stvar.benchmark.bed" ./confident.bed
! rm -rf ./filtered/ ; mkdir ./filtered/

In [ ]:
# Stratifying by SVTYPE and SVLEN
SV_TYPES=["del", "ins", "dup", "insdup", "inv"]
REGION_STRINGS=["--regions-file confident.bed --regions-overlap record", " "]  # Same overlap criterion as training script
REGION_IDS=["confident BED", "whole genome"]
for t in SV_TYPES:
    if t == "dup" or t == "insdup" or t == "inv":
        test_chr = lambda callset: np.isin(callset['variants/CHROM'],['chr1', 'chr3', 'chr5', 'chr7', 'chr9', 'chr11', 'chr13', 'chr15', 'chr17', 'chr19', 'chr21'])
        test_chr_label = "odd chrs"
    else:
        test_chr = lambda callset: np.isin(callset['variants/CHROM'],['chr1', 'chr2', 'chr3'])
        test_chr_label = "chr1..3"
    for r in [0, 1]:
        # 1. Plotting ROC curves for different SVLEN ranges
        MIN=10_000
        MAX=2_000_000
        ! bcftools filter {REGION_STRINGS[r]} --include 'SVLEN>'{MIN}' && SVLEN<='{MAX} --output-type z {t}_all_score.vcf.gz --output filtered/{t}_region{r}_10k_2m.vcf.gz
        ! tabix filtered/{t}_region{r}_10k_2m.vcf.gz
        MIN=10_000
        MAX=25_000
        ! bcftools filter {REGION_STRINGS[r]} --include 'SVLEN>'{MIN}' && SVLEN<='{MAX} --output-type z {t}_all_score.vcf.gz --output filtered/{t}_region{r}_10k_25k.vcf.gz
        ! tabix filtered/{t}_region{r}_10k_25k.vcf.gz
        MIN=25_000
        MAX=100_000 
        ! bcftools filter {REGION_STRINGS[r]} --include 'SVLEN>'{MIN}' && SVLEN<='{MAX} --output-type z {t}_all_score.vcf.gz --output filtered/{t}_region{r}_25k_100k.vcf.gz
        ! tabix filtered/{t}_region{r}_25k_100k.vcf.gz
        MIN=100_000
        MAX=2_000_000 
        ! bcftools filter {REGION_STRINGS[r]} --include 'SVLEN>'{MIN}' && SVLEN<='{MAX} --output-type z {t}_all_score.vcf.gz --output filtered/{t}_region{r}_100k_2m.vcf.gz
        ! tabix filtered/{t}_region{r}_100k_2m.vcf.gz
        series_vcfs = [f"filtered/{t}_region{r}_10k_2m.vcf.gz", f"filtered/{t}_region{r}_10k_25k.vcf.gz", f"filtered/{t}_region{r}_25k_100k.vcf.gz", f"filtered/{t}_region{r}_100k_2m.vcf.gz"]
        series_names = ["10k..2m", "10k..25k", "25k..100k", "100k..2m"]
        callsets = []
        for i, vcf in tqdm.tqdm(enumerate(series_vcfs)):
            callset = allel.read_vcf(vcf, fields=['variants/SCORE',
                        'variants/CALIBRATION_SENSITIVITY',
                        'variants/training',
                        'variants/extracted',
                        'variants/CHROM',
                        'variants/SUPP_PBSV',
                        'variants/SUPP_SNIFFLES',
                        'variants/SUPP_PAV'
                    ])
            callsets.append(callset)
        fig, ax = plt.subplots(figsize=(7, 7))
        num_vcf = len(callsets)
        colors = cm.get_cmap('tab10', num_vcf).colors
        for i, callset in tqdm.tqdm(enumerate(callsets), total=num_vcf):
            vcf_label = series_names[i]
            print(f"Processing Index {i}: Labeling curve as '{vcf_label}' File name: '{series_vcfs[i]}'")
            metrics_mask = test_chr(callset) & ~np.isnan(callset['variants/SCORE'])
            if np.any(metrics_mask):
                fpr, tpr, _ = sklearn.metrics.roc_curve(
                    y_true=callset['variants/training'][metrics_mask], 
                    y_score=callset['variants/SCORE'][metrics_mask]
                )
                roc_auc = sklearn.metrics.auc(fpr, tpr)
                ax.plot(fpr, tpr, label=f"{vcf_label} (AUC={roc_auc:.3f})", color=colors[i], lw=2, alpha=0.8)    

        # 2. Plotting points for individual callers
        callset = callsets[0]
        y_true_all = callset['variants/training'][test_chr(callset)]

        gt_mask = (callset['variants/SUPP_PBSV'] > 0) & test_chr(callset)
        y_pred_gt = gt_mask[test_chr(callset)].astype(int) 
        tn, fp, fn, tp = sklearn.metrics.confusion_matrix(y_true_all, y_pred_gt).ravel()
        point_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        point_tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        ax.scatter(point_fpr, point_tpr, color='white', marker='s', s=100, edgecolors='black', zorder=6, label=f"SUPP pbsv")
    
        gt_mask = (callset['variants/SUPP_SNIFFLES'] > 0) & test_chr(callset)
        y_pred_gt = gt_mask[test_chr(callset)].astype(int) 
        tn, fp, fn, tp = sklearn.metrics.confusion_matrix(y_true_all, y_pred_gt).ravel()
        point_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        point_tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        ax.scatter(point_fpr, point_tpr, color='white', marker='^', s=100, edgecolors='black', zorder=6, label=f"SUPP Sniffles")
    
        gt_mask = (callset['variants/SUPP_PAV'] > 0) & test_chr(callset)
        y_pred_gt = gt_mask[test_chr(callset)].astype(int) 
        tn, fp, fn, tp = sklearn.metrics.confusion_matrix(y_true_all, y_pred_gt).ravel()
        point_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        point_tpr = tp / (tp + fn) if (tp + fn) > 0 else 0

        # 3. Finalizing the plot
        ax.scatter(point_fpr, point_tpr, color='white', marker='o', s=100, edgecolors='black', zorder=6, label=f"SUPP PAV")
        ax.plot([0, 1], [0, 1], color='grey', linestyle='--', alpha=0.5) # Diagonal baseline
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(f"{t}, {test_chr_label}, {REGION_IDS[r]}, 292 HPRC and HGSVC samples, GRCh38")
        ax.legend(loc='lower right', fontsize='small', frameon=True)
        ax.grid(True, linestyle=':', alpha=0.6)
        plt.tight_layout()
        plt.show()

In [ ]:
# def plot_roc(callsets, calibration_sensitivity_thresholds=[0.7,0.9]):
#     # Masks
#     # Remark: we trained on chr6..Y. We test here on chr1..5.
#     test_mask_lambda = lambda callset: ((callset['variants/CHROM'] == 'chr1') | 
#                                         (callset['variants/CHROM'] == 'chr2') | 
#                                         (callset['variants/CHROM'] == 'chr3') | 
#                                         (callset['variants/CHROM'] == 'chr4') | 
#                                         (callset['variants/CHROM'] == 'chr5'))
    
#     fig = plt.figure(figsize=(5, 5))
#     for mask_title, mask_color, additional_mask_lambda in [
#                                                ['all', 'blue',
#                                                 lambda callset: test_mask_lambda(callset) ]]:
        
#         for i, callset in tqdm.tqdm(enumerate(callsets)):
#             additional_mask = additional_mask_lambda(callset)
            
#             # Score ROC
#             metrics_mask = ~np.isnan(callset['variants/SCORE']) & additional_mask
#             fpr, tpr, thresholds = sklearn.metrics.roc_curve(
#                 y_true=callset['variants/training'][metrics_mask], 
#                 y_score=callset['variants/SCORE'][metrics_mask])
#             plt.plot(fpr, tpr, label=mask_title if i==0 else None, c=mask_color, alpha=0.25)
            
#             # CAL_SENS thresholds
#             metrics_mask = ~np.isnan(callset['variants/CALIBRATION_SENSITIVITY']) & additional_mask
#             fpr, tpr, thresholds = sklearn.metrics.roc_curve(
#                 y_true=callset['variants/training'][metrics_mask], 
#                 y_score=1 - callset['variants/CALIBRATION_SENSITIVITY'][metrics_mask])
#             fpr_selection = fpr[[np.argmax(tpr >= cst) for cst in calibration_sensitivity_thresholds]]
#             tpr_selection = tpr[[np.argmax(tpr >= cst) for cst in calibration_sensitivity_thresholds]]
#             plt.scatter(fpr_selection, tpr_selection, label=None, 
#                         c=mask_color, marker='d', s=4)

#     plt.xlabel('False Positive Rate')
#     plt.ylabel('True Positive Rate')
#     plt.title(f'Deletions >=10kb, chr1..5, 145 HPRC+HGSVC samples, GRCh38.')
#     plt.legend()
#     plt.show()
